# Building a Multimodal Agent with LlamaCpp and Strands SDK

This notebook demonstrates how to create agents using LlamaCpp with Strands SDK. You'll learn to use quantized models locally with advanced features like grammar constraints, multimodal processing, and custom tools.

LlamaCpp supports any GGUF-format quantized model. You can easily switch between models by downloading different GGUF files and updating the model path. Popular options include Llama, Mistral, Phi, and Qwen families. Simply change the model file in your server command to use a different model:

## What You'll Learn

- Running quantized models locally with LlamaCpp
- Using grammar constraints for controlled output
- Processing multimodal inputs (audio and vision)
- Configuring advanced sampling parameters
- Integrating custom tools with your agent

## Setup and Installation

For this tutorial, we use Qwen2.5-Omni for its multimodal capabilities (audio + vision + text).

Before running this notebook, ensure you have:

1. **Python 3.8+** installed
2. **llama.cpp** with server support ([Installation Guide](https://github.com/ggerganov/llama.cpp))
3. **Model files** downloaded:

```bash
# Download Qwen2.5-Omni model files
mkdir -p models && cd models

# Main model (4.68 GB)
huggingface-cli download ggml-org/Qwen2.5-Omni-7B-GGUF \
  Qwen2.5-Omni-7B-Q4_K_M.gguf --local-dir .

# Multimodal projector (1.55 GB) - Required for audio/vision
huggingface-cli download ggml-org/Qwen2.5-Omni-7B-GGUF \
  mmproj-Qwen2.5-Omni-7B-Q8_0.gguf --local-dir .

cd ..
```

4. **Start the server**:

```bash
llama-server -m models/Qwen2.5-Omni-7B-Q4_K_M.gguf \
  --mmproj models/mmproj-Qwen2.5-Omni-7B-Q8_0.gguf \
  --host 0.0.0.0 --port 8080 -c 8192 -ngl 50 --jinja
```

## Install Python Dependencies

Install the Strands SDK and required libraries for audio processing, image handling, and notebook widgets.

In [ ]:
# Install dependencies from requirements.txt
!pip install -q -r requirements.txt

## Import Required Libraries

Import the Strands SDK components and utility functions we'll use throughout this tutorial. The utils folder contains helper functions for audio, image, grammar, and benchmarking operations.

In [ ]:
import os
import sys
import json
from datetime import datetime
from typing import List, Dict, Any
from pathlib import Path

# Add utils to path
utils_path = os.path.join(os.getcwd(), 'utils')
if utils_path not in sys.path:
    sys.path.append(utils_path)

# Import Strands SDK
from strands import Agent, tool
from strands.models.llamacpp import LlamaCppModel
from pydantic import BaseModel, Field

# Import utilities
from utils import (
    # Audio utilities
    SimpleAudioRecorder, create_simple_audio_interface, display_simple_audio_interface,
    
    # Image utilities  
    create_test_image, create_complex_test_image, image_to_bytes,
    analyze_image_with_llamacpp, create_image_analysis_demo,
    
    # Grammar and sampling utilities
    demonstrate_grammar_constraint, test_sampling_strategy,
    get_predefined_grammars, get_sampling_strategies,
    run_grammar_constraints_demo, run_sampling_strategies_demo,
    test_structured_output,
    
    # Benchmark utilities
    benchmark_performance, analyze_benchmark_results, visualize_performance,
    run_comprehensive_benchmark
)

# IPython for multimedia display
from IPython.display import Audio, Image as IPImage, display, HTML
import ipywidgets as widgets

import os
import sys
import json
from datetime import datetime
from typing import List, Dict, Any
from pathlib import Path

# Add utils to path
utils_path = os.path.join(os.getcwd(), 'utils')
if utils_path not in sys.path:
    sys.path.append(utils_path)

# Import Strands SDK
from strands import Agent, tool
from strands.models.llamacpp import LlamaCppModel
from pydantic import BaseModel, Field

# Import utilities
from utils import (
    # Audio utilities
    AudioRecorder, create_audio_interface, display_audio_interface,
    
    # Image utilities  
    create_test_image, image_to_bytes, analyze_image_with_llamacpp,
    
    # Grammar and sampling utilities
    demonstrate_grammar_constraint, test_sampling_strategy,
    get_predefined_grammars, get_sampling_strategies,
    
    # Benchmark utilities
    benchmark_performance, run_comprehensive_benchmark
)

# IPython for multimedia display
from IPython.display import Audio, Image as IPImage, display, HTML
import ipywidgets as widgets

### Define Structured Data Models

First, define Pydantic models that will be used for type-safe structured output generation. These models ensure the AI generates data in exactly the format you need.

In [ ]:
# Define structured output models
class TaskPlan(BaseModel):
    """A structured task plan."""
    title: str = Field(description="Brief title of the task")
    steps: List[str] = Field(description="List of steps to complete")
    estimated_time: int = Field(description="Estimated time in minutes")
    difficulty: str = Field(description="Easy, Medium, or Hard")
    
class ProductReview(BaseModel):
    """A structured product review."""
    product_name: str = Field(description="Name of the product")
    rating: int = Field(description="Rating from 1 to 5")
    pros: List[str] = Field(description="Positive aspects")
    cons: List[str] = Field(description="Negative aspects")
    recommendation: bool = Field(description="Would you recommend it?")

## Advanced Sampling Parameters

LlamaCpp offers fine-grained control over text generation through various sampling strategies. The following examples demonstrate how different parameters affect output quality and creativity.

In [ ]:
# Test different sampling strategies
strategies = get_sampling_strategies()
prompt = "Write a creative story opening about a mysterious door:"

# Test first 3 strategies
for strategy in strategies[:3]:
    test_sampling_strategy(
        params=strategy["params"],
        name=strategy["name"],
        prompt=prompt
    )

### Grammar Constraints in Action

Test predefined GBNF grammars that force specific output formats. Watch how the model's responses are constrained to match exact patterns like yes/no, multiple choice, or JSON structures.

In [ ]:
# Demonstrate various grammar constraints
grammars = get_predefined_grammars()

# Test a few interesting examples
examples_to_test = ["yes_no", "multiple_choice", "simple_json", "color_names"]

for grammar_name in examples_to_test:
    if grammar_name in grammars:
        grammar_info = grammars[grammar_name]
        
        demonstrate_grammar_constraint(
            grammar=grammar_info["grammar"],
            prompt=grammar_info["example_prompt"],
            description=f"{grammar_name.upper()}: {grammar_info['description']}",
            base_url="http://localhost:8080",
            temperature=0.1,
            max_tokens=50
        )

### Custom Grammar Examples

Create your own GBNF grammar for specific use cases and use JSON schemas as an alternative constraint method. Both approaches guarantee structured output without post-processing.

In [ ]:
# Example: Create a custom grammar for star ratings
star_rating_grammar = '''root ::= rating " stars"
rating ::= "1" | "2" | "3" | "4" | "5"'''

# Create model with custom grammar
model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={"temperature": 0.1, "max_tokens": 20}
)

# Apply the grammar constraint
model.use_grammar_constraint(star_rating_grammar)
agent = Agent(model=model)

# Test the constraint
test_prompts = [
    "How would you rate this restaurant?",
    "What's your opinion on this movie?",
    "Rate the customer service experience:"
]

for prompt in test_prompts:
    response = agent(prompt)
    print(f"{prompt} -> {response}")

# Example: JSON Schema constraint
json_model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={"temperature": 0.3, "max_tokens": 100}
)

# Define JSON schema for structured output
product_schema = {
    "type": "object",
    "properties": {
        "product_name": {"type": "string"},
        "price": {"type": "number", "minimum": 0},
        "category": {"type": "string", "enum": ["electronics", "clothing", "food", "books"]},
        "in_stock": {"type": "boolean"}
    },
    "required": ["product_name", "price", "category", "in_stock"]
}

json_model.use_json_schema(product_schema)
json_agent = Agent(model=json_model)

response = json_agent("Generate information for a laptop product:")

## Custom Tools

Extend your agent's capabilities by defining custom functions that the model can call. The following tools demonstrate how to add domain-specific functionality to your agent.

In [ ]:
# Define custom tools
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> Dict[str, Any]:
    """
    Calculate Body Mass Index (BMI).
    
    Args:
        weight_kg: Weight in kilograms
        height_m: Height in meters
        
    Returns:
        BMI value and category
    """
    bmi = weight_kg / (height_m ** 2)
    
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    
    return {
        "bmi": round(bmi, 2),
        "category": category,
        "healthy_range": "18.5 - 24.9"
    }

@tool
def get_weather_description(condition: str) -> str:
    """
    Get a poetic description of weather conditions.
    
    Args:
        condition: Weather condition (sunny, rainy, cloudy, etc.)
        
    Returns:
        Poetic weather description
    """
    descriptions = {
        "sunny": "Golden rays dance across azure skies",
        "rainy": "Silver droplets paint the world anew",
        "cloudy": "Cotton castles drift through endless blue",
        "snowy": "Crystal blankets hush the sleeping earth"
    }
    
    return descriptions.get(condition.lower(), f"The weather shows its {condition} face")

### Create Agent with Tools

Initialize an agent with access to your custom tools. The agent will automatically determine when to use these tools based on the user's query.

In [ ]:
# Create agent with tools
model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={
        "temperature": 0.7,
        "max_tokens": 300,
        "top_k": 40
    }
)

agent = Agent(
    model=model,
    tools=[calculate_bmi, get_weather_description],
    system_prompt="You are a helpful assistant with access to calculation and description tools."
)

### Test Tool Usage

Observe how the agent intelligently calls the appropriate tools based on natural language queries. The agent handles both single and compound tool requests seamlessly.

In [ ]:
# Test tool usage
test_queries = [
    "What's the BMI for someone who is 1.75m tall and weighs 70kg?",
    "Give me a poetic description of rainy weather",
    "Calculate BMI for 85kg and 1.80m, then describe sunny weather"
]

for query in test_queries:
    response = agent(query)
    print(f"Q: {query}")
    print(f"A: {response}\n")

## Multimodal Processing

Process audio and images alongside text using Qwen2.5-Omni's multimodal capabilities:

### Audio Processing - Speech Recognition & Translation

Use Qwen2.5-Omni's native audio capabilities to transcribe speech in any language and automatically translate to English. The interface below provides an interactive way to test multilingual speech recognition.

In [14]:
# Create speech recognition interface
recorder = SimpleAudioRecorder(sample_rate=16000)

# Create interface for multilingual speech recognition
interface_components = create_simple_audio_interface(
    recorder=recorder,
    base_url="http://localhost:8080"
)

# Display the interface
display_simple_audio_interface(interface_components)

IntProgress(value=0, bar_style='info', description='Progress:', layout=Layout(visibility='hidden', width='100%…

Output(layout=Layout(height='50px'))

Output(layout=Layout(height='200px', overflow='auto'))

1. Original transcription: '你好，我明天去日本。'
2. Language detected: Chinese
3. English translation: 'Hello, I'm going to Japan tomorrow.'

 Example: Audio message format for Qwen2.5-Omni

```python
example_message = {
    "role": "user",
    "content": [
        {
            "type": "audio",
            "audio": {
                "data": "base64_encoded_audio_data_here",
                "format": "wav"
            }
        },
        {
            "type": "text", 
            "text": "Please transcribe exactly what was said. If not in English, provide: 1) Original transcription 2) Language detected 3) English translation"
        }
    ]
}
```
 The SDK handles this formatting automatically when using the interface above

In [ ]:
# Create and analyze test image
test_image = create_test_image()
display(test_image)

# Analyze image
analysis = analyze_image_with_llamacpp(
    test_image,
    "Describe this image in detail. What shapes and colors do you see?",
    max_tokens=200
)

## Performance Optimization

Compare three optimization strategies to understand the trade-offs between quality, speed, and resource usage. Each configuration is tailored for different production scenarios.

In [ ]:
# Test different performance optimization settings

# Configuration 1: High Quality (slower)
high_quality_model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={
        "temperature": 0.3,
        "top_k": 10,
        "repeat_penalty": 1.2,
        "max_tokens": 100
    }
)

# Configuration 2: Balanced Performance
balanced_model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={
        "temperature": 0.7,
        "top_k": 40,
        "min_p": 0.05,
        "max_tokens": 100,
        "cache_prompt": True  # Enable prompt caching
    }
)

# Configuration 3: Speed Optimized
speed_model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={
        "temperature": 0.8,
        "top_k": 20,
        "max_tokens": 100,
        "cache_prompt": True,
        "n_probs": 0  # Disable probability computation
    }
)

# Test each configuration
prompt = "Explain machine learning in simple terms:"

agent_hq = Agent(model=high_quality_model)
response_hq = agent_hq(prompt)

agent_balanced = Agent(model=balanced_model)
response_balanced = agent_balanced(prompt)

agent_speed = Agent(model=speed_model)
response_speed = agent_speed(prompt)

### Performance Benchmark

Run a comprehensive benchmark to measure response times and quality across different configurations. This data helps you choose optimal settings for your specific use case.

In [ ]:
# Run comprehensive performance benchmark
benchmark_results = run_comprehensive_benchmark(base_url="http://localhost:8080")

## Next Steps

This notebook demonstrated the key features of LlamaCpp with Strands SDK. You can now:

- Experiment with different GGUF models from Hugging Face
- Create custom grammars for your specific use cases
- Build production applications with local AI
- Explore multimodal capabilities with other models

For more examples and documentation, visit the [Strands SDK Documentation](https://docs.strands.ai).